# Sampling module

**What's in this notebook?** This notebook illustrates the sampling infrastructure within JAXVacua. The `data_sampler` class provides flexible methods for generating initial guesses for the moduli and fluxes, which serve as starting points for the numerical minimisation described in notebooks [7](7_ISD_sampling_flux_vacua.ipynb) and [8](8_ISD_sampling_wrapper.ipynb).

**In this notebook, you will learn:**
- How to create a `data_sampler` object and configure its bounds
- The different moduli sampling modes: `"box"`, `"cone"`, `"stretched_cone"`, `"tip_ray"`, `"random_ray"`, `"random_rays"`
- How ISD sampling works: replacing a subset of the fluxes by solving the ISD condition $\star G_3 = \mathrm{i}\, G_3$ for given moduli
- The four ISD modes (`"H"`, `"F"`, `"ISD+"`, `"ISD-"`) and when to use each

**Prerequisites:** Basic JAXVacua usage ([notebook 2](../01_basics/2_jaxvacua_overview.ipynb)), flux vacua finding ([notebook 5](5_finding_flux_vacua.ipynb)).

**Related notebooks:** [7_ISD_sampling_flux_vacua](7_ISD_sampling_flux_vacua.ipynb) (single-vacuum ISD example), [8_ISD_sampling_wrapper](8_ISD_sampling_wrapper.ipynb) (large-scale sampling), [9_sampling_vacua_from_fluxes](9_sampling_vacua_from_fluxes.ipynb) (flux-first approach).

(*Created:* Andreas Schachner, June 25, 2024)

## Imports

In [ ]:
# General imports
import warnings
import time
import numpy as np
from tqdm.auto import tqdm

# JAX imports
from jax import vmap
import jax
import jax.numpy as jnp
jax.config.update("jax_enable_x64", True)

# Plotting
import seaborn as sn
import matplotlib.pyplot as plt
cmap = sn.color_palette("viridis", as_cmap=True)

# JAXVacua
import jaxvacua as jvc

warnings.filterwarnings('ignore')

## Sampling methods at $h^{1,2}=2$

Load model from files

In [ ]:
#if False:
if True:
    h12=2
    model = jvc.FluxVacuaFinder(h12 = h12, Q=276, model_ID = 1, maximum_degree = 2,limit="LCS",model_type="KS")

Alternatively, use CYTools

In [ ]:
from cytools import Polytope, Cone, fetch_polytopes, read_polytopes

if False:
    p = fetch_polytopes(h11=2,h12=272,limit=5,lattice="N",as_list=True)[0]
    cy = p.triangulate().get_cy()
    mcap = cy.mori_cone_cap(in_basis=True)
    Kcup = mcap.dual_cone()
    basis_trafo = Kcup.extremal_rays()

    model = jvc.FluxEFT(h12=cy.h11(), Q=cy.h11()+cy.h12()+2, model_type="KS", maximum_degree=2, 
                                      use_cytools=True, mirror_cy = cy, basis_transformation=basis_trafo)

As objective function, we take the $F$-terms for the moduli as computed in `model.DW`.
For later purposes, we vectorise this function by using `jax.vmap`:

In [ ]:
DW_v = vmap(model.DW)

It is also convenient to introduce a data sampler that constrains our sampling procedure to specific regions in moduli and flux space:

In [ ]:
sampler = jvc.data_sampler(model,flux_bounds=[-5,5],moduli_bounds=[0,5],axion_bounds=[-0.5,0.5])

In [ ]:
seed = 42
rns_key = jvc.PRNGSequence(seed)

sampler_jax = jvc.data_sampler(model,flux_bounds=[-5,5],moduli_bounds=[0,5],axion_bounds=[-0.5,0.5],use_jax=True,seed=seed)

moduli,tau,fluxes = sampler_jax.initial_guesses(5,rns_key=rns_key)
moduli,tau,fluxes

In [ ]:
#%%timeit
moduli,tau,fluxes = sampler_jax.initial_guesses_ISD(500)
#moduli,tau,fluxes

In [ ]:
#%%timeit
moduli,tau,fluxes = sampler.initial_guesses_ISD(500)
#moduli,tau,fluxes

In [ ]:
#%%timeit
#rns_key = jvc.PRNGSequence(seed)
moduli_new,tau_new,fluxes_new = sampler_jax.initial_guesses_ISD(500)
moduli_new[:5]

In [ ]:
#%%timeit
moduli_new,tau_new,fluxes_new = sampler.initial_guesses_ISD(500)
moduli_new[:5]

In [ ]:
#?sampler_jax.initial_guesses_ISD

In [ ]:
moduli_new,tau_new,fluxes_new = sampler_jax.initial_guesses_ISD(5000,filter_moduli=True,vmap_dim=10000)
moduli_new.shape

In [ ]:
moduli_new,tau_new,fluxes_new = sampler.initial_guesses_ISD(5000,filter_moduli=True,vmap_dim=10000)

For reproducability, we use a seed for random number generation

In [ ]:
seed = 42
np.random.seed(seed)
np.random.rand(seed)

Initial guesses can then be obtained as follows:

In [ ]:
moduli,tau,fluxes = sampler.initial_guesses(5)
moduli,tau,fluxes

In [ ]:
sampler.update_interior_points(num_pts = 2000)

In [ ]:
len(sampler._cone_points),len(np.unique(sampler._cone_points,axis=0))

Compute shortest distance to next point

In [ ]:
pts = sampler._cone_points
norms = np.linalg.norm(pts,axis=1)

In [ ]:
from scipy import spatial

eps = 0.1
dat = []
for i,pt in tqdm(enumerate(pts)):
    norm = norms[i]
    pts0 = np.delete(pts.copy(), i, 0)
    norms0 = np.delete(norms.copy(), i, 0)
    
    flag = ((norms0>norm-eps)&(norms0<norm+eps))
    pts0 = pts0[flag]
    
    if len(pts0)==0:
        
        pts0 = np.delete(pts.copy(), i, 0)
        norms0 = np.delete(norms.copy(), i, 0)
        
        flag = ((norms0>norm-10*eps)&(norms0<norm+10*eps))
        
        pts0 = pts0[flag]
        
    
    #sys.exit()
    
    distance,index = spatial.KDTree(pts0).query(pt)
    
    if np.isinf(distance):
        sys.exit()
    
    dat.append([distance,index])
    
dat = np.array(dat)

In [ ]:
dis = dat[:,0]
np.mean(dis),np.min(dis),np.max(dis)

In [ ]:
np.where(dis==np.max(dis))

In [ ]:
np.where(dis==np.min(dis))

In [ ]:
dat[:,1][14805]

In [ ]:
pts[14805],pts[17603]

In [ ]:
dat

In [ ]:
len(pts)

### Sampling moduli values

As a quick illustration, we can compare the following ways of sampling moduli.
The default option for sampling modes `"cone"` and `"stretched_cone"` is using a linear solver from `gurobi` to solve the hyperplane constraints $H\cdot x\geq 0$.

In [ ]:
sampling_modes = ["box","cone","stretched_cone","tip_ray","random_ray","random_rays"]
fig = plt.figure(dpi=150,figsize=(8,6))
for sampling_mode in sampling_modes:
    
    stretching = 0.
    if sampling_mode=="stretched_cone":
        stretching = 1.
        
    moduli = sampler.get_moduli(1000, 
                    sampling_mode = sampling_mode,
                    stretching  = stretching,
                    n_rays = 2,
                    perturbation = 1,
                    use_rays = False)
    
    sn.scatterplot(x=moduli[:,0],y=moduli[:,1],label=sampling_mode)
    
plt.xlabel(r"Im$(z^1)$")
plt.ylabel(r"Im$(z^2)$")
plt.title(r"Sampling using linear programming")
plt.show()

In [ ]:
from cytools import Polytope
import numpy as np

p = Polytope(np.array([[ 0,  0, -1,  0,  0,  0,  1,  1, -1,  0],
       [ 0,  0,  0,  0,  1, -1, -1,  0,  0, -1],
       [ 1, -1,  0,  0,  0, -1,  0,  0,  0, -1],
       [ 0, -1,  1,  1,  0,  0,  0,  0,  0, -1]]).T)

p.chi("N")

**different norms**

In [ ]:
sampling_modes = ["box","cone","stretched_cone","tip_ray","random_ray","random_rays"]

for norm in ["l1","l2","inf"]:
    fig = plt.figure(dpi=100,figsize=(6,4))
    for sampling_mode in sampling_modes:

        stretching = 0.
        if sampling_mode=="stretched_cone":
            stretching = 1.

        moduli = sampler.get_moduli(1000, 
                        sampling_mode = sampling_mode,
                        stretching  = stretching,
                        n_rays = 2,
                        perturbation = 1,
                        use_rays = False)
        
        moduli = sampler.rescale_points(moduli,norm=norm,maxval=2)

        sn.scatterplot(x=moduli[:,0],y=moduli[:,1],label=sampling_mode)

    plt.xlabel(r"Im$(z^1)$")
    plt.ylabel(r"Im$(z^2)$")
    plt.title(r"Sampling using "+norm+" norm")
    plt.show()

We can also use rays for sampling modes `"cone"` and `"stretched_cone"`:

In [ ]:
sampling_modes = ["box","cone","stretched_cone","tip_ray","random_ray","random_rays"]
fig = plt.figure(dpi=150,figsize=(8,6))
for sampling_mode in sampling_modes:
    
    stretching = 0.
    if sampling_mode=="stretched_cone":
        stretching = 1.
        
    moduli = sampler.get_moduli(1000, 
                    sampling_mode = sampling_mode,
                    stretching  = stretching,
                    n_rays = 2,
                    perturbation = 1,
                    use_rays = True)
    
    sn.scatterplot(x=moduli[:,0],y=moduli[:,1],label=sampling_mode)
    
plt.xlabel(r"Im$(z^1)$")
plt.ylabel(r"Im$(z^2)$")
plt.title(r"Sampling using rays")
plt.show()

Here, we sample
* fluxes uniformly from a box set by the input parameter `flux_bounds` above,
* $\tau$ uniformly from its standard fundamental domain: $|\tau|\geq 1$, $|\text{Re}(\tau)|\leq 0.5$,
* $\text{Re}(z^i)$ uniformly from box as set by `axion_bounds` above,
* $\text{Im}(z^i)$ either uniformly from a box or the mirror Kähler cone (uniformly on the coefficients of the generators/rays)

**WARNING:** This is a really superficial sampling class that is not quite flexible enough for different

## ISD sampling

### General principle of ISD sampling

The general concept behind *ISD sampling* is that, instead of randomly choosing integer fluxes from some distribution, 
a subset of the fluxes can be replaced in favour of fixing values for the moduli, see in particular [2306.06160](https://arxiv.org/pdf/2306.06160).
In the process, the remaining fluxes can be fixed through the ISD condition $\star G_3 = \text{i}G_3$ where the Hodge-$\star$ depends explicitily on the moduli.
A clear advantage is that the ISD condition is **linear** in the fluxes and typically can be easily solved for given inputs.
(In contrast, when fixing all fluxes and solving the ISD condition for the moduli, one has to deal with highly non-trivial coupled equations in the moduli which can typically only be solved numerically.)
However, one immediate disadvantage is that the fluxes sampled through the ISD condition are typically not **quantised**.
But even after rounding the fluxes to integers, the chosen parameters approximately solve the ISD condition and hence serve as valuable starting guesses in a numerical search for flux vacua.

Below, we demonstrate the above idea in one single example before setting up a larger scan further below.
Let us choose initial guesses `z0` and `tau0` for the moduli $z^i_0$ and the axio-dilaton $\tau_0$
together with a choice `Hflux` of NSNS flux quanta $h$

In [ ]:
# Moduli starting guesses
z0 = jnp.array([0.3+3j , 0.36+3.1j])

# Axio-dilaton starting guess
tau0 = -0.3+6.7j

# Choices of H-fluxes
Hflux = jnp.array([39., -13.,  -4.,   0.,  -0.,  0.])

Then, the RR-fluxes $f$ at the minimum are specified through the following version of the ISD condition

$$
    f=(s\,  M(z_0^i,\overline{z}_0^i)\Sigma + c_0)\, h\; ,\quad \tau_0=c_0 + \text{i} s \, .
$$

This particular version of ISD sampling was employed in [2501.03984](https://arxiv.org/abs/2501.03984).
Here, the ISD -matrix $M$ is computed from `model.ISD_matrix`:

In [ ]:
s = tau0.imag
c0 = tau0.real
M = model.ISD_matrix(z0,jnp.conj(z0))

Fflux = jnp.matmul(M,jnp.matmul(model.periods.sigma(),Hflux))*s+c0*Hflux

Fflux.real

As expected, the fluxes obtained in this way are **not quantised**. 
For these values of the fluxes, the ISD condition or equivalently $D_IW=0$ is satisfied:

In [ ]:
model.DW(z0,jnp.conj(z0),tau0,jnp.conj(tau0),jnp.append(Fflux.real,Hflux))

To find an actual flux vacuum, we now round this choice to integers leading to

In [ ]:
Fflux_rounded = jnp.around(Fflux.real,0)
fluxes0 = jnp.append(Fflux_rounded,Hflux)
fluxes0

The above steps have been collected in a single wrapper function `model.ISD_sampling` (see below for additional details)

In [ ]:
fluxes0 = sampler.ISD_sampling(z0,jnp.conj(z0),tau0,jnp.conj(tau0),Hflux,mode="H",output="full",return_integer_flux=True).real
fluxes0

However, since we changed the choices of RR-fluxes, the initial guesses `z0` and `tau0` do not correspond tothe acutal points
in moduli space at which the scalar potential is minimised as can be seen by computing $D_I W$:

In [ ]:
model.DW(z0,jnp.conj(z0),tau0,jnp.conj(tau0),fluxes0)

Nonetheless, the initial guesses `z0` and `tau0` are typically very close to an actual solution to $D_I W=0$
which we can e.g. find by employing Newton's method.

This is illustrated in the notebook [7_ISD_sampling_flux_vacua.ipynb](7_ISD_sampling_flux_vacua.ipynb)

### Different versions of ISD sampling

Lastly, let describe the four modi to solve the ISD condition for a given choices of half of the fluxes.
Either we use the following form of the ISD condition

$$
    f_1-\tau h_1=\overline{\mathcal{N}}(z^i,\overline{z}^i)\, (f_2-\tau h_2)\, ,
$$

in terms of the gauge kinetic matrix $\mathcal{N}$ which can be computed using `model.gauge_kinetic_matrix`.
This expression can be solved for fluxes $(f_1,h_1)$ (corresponding to `mode="ISD+"`) 
or $(f_2,h_2)$ (associated to `mode="ISD-"`) corresponding to the components of the 
RR-flux vector $f=(f_1,f_2)$ and the NSNS-flux vector $h=(h_1,h_2)$
for given input values for the moduli $z^i$ and the axio-dilaton $\tau$.
The other two modi are obtained by rewriting the above expression in the following form

$$
    f=(s \, M(z^i,\overline{z}^i)\Sigma +  c_0)\, h\; ,\quad \tau=c_0 + \text{i} s \, .
$$

This can then be solved for RR-flux vector $f=(f_1,f_2)$ (`mode="F"`) 
or the NSNS-flux vector $h=(h_1,h_2)$ (`mode="H"`).

Using the same choice of fluxes and initial guesses as above, let us first collect

**Summary of ISD sampling modes:**

| Mode | Fixed input | Solved for | ISD condition used |
|------|-------------|------------|--------------------|
| `"H"` | NSNS fluxes $h = (h_1, h_2)$ | RR fluxes $f$ | $f = (s\,M\Sigma + c_0)\,h$ |
| `"F"` | RR fluxes $f = (f_1, f_2)$ | NSNS fluxes $h$ | $h = (s\,M\Sigma + c_0)^{-1}\,f$ |
| `"ISD+"` | $(f_2, h_2)$ | $(f_1, h_1)$ | $f_1 - \tau h_1 = \bar{\mathcal{N}}\,(f_2 - \tau h_2)$ |
| `"ISD-"` | $(f_1, h_1)$ | $(f_2, h_2)$ | $f_2 - \tau h_2 = \bar{\mathcal{N}}^{-1}(f_1 - \tau h_1)$ |

Here $M$ is the ISD matrix (`model.ISD_matrix`), $\mathcal{N}$ is the gauge kinetic matrix (`model.gauge_kinetic_matrix`), $s = \mathrm{Im}(\tau)$, and $c_0 = \mathrm{Re}(\tau)$.
Modes `"H"` and `"F"` tend to give the best initial guesses because the ISD matrix $M$ is computed at the exact input moduli values.

In [ ]:
h1 = jnp.array([39., -13.,  -4.])
h2 = jnp.array([ 0.,  0.,  0.])
f1 = jnp.array([4.,  -3.,  -2.])
f2 = jnp.array([-2.,   3.,   2.])

# Choices of H-fluxes
Hflux = jnp.append(h1,h2)

# Choices of F-fluxes
Fflux = jnp.append(f1,f2)

# Choice for ISD+
ISDflux_plus = jnp.append(f2,h2)

# Choice for ISD-
ISDflux_minus = jnp.append(f1,h1)

Then the four different versions of ISD sampling can be used as follows

In [ ]:
fluxes0 = sampler.ISD_sampling(z0,jnp.conj(z0),tau0,jnp.conj(tau0),Hflux,mode="H",output="full",return_integer_flux=True).real
fluxes0,np.sum(np.abs(model.DW(z0,jnp.conj(z0),tau0,jnp.conj(tau0),fluxes0)))

In [ ]:
fluxes0 = sampler.ISD_sampling(z0,jnp.conj(z0),tau0,jnp.conj(tau0),Fflux,mode="F",output="full",return_integer_flux=True).real
fluxes0,np.sum(np.abs(model.DW(z0,jnp.conj(z0),tau0,jnp.conj(tau0),fluxes0)))

In [ ]:
fluxes0 = sampler.ISD_sampling(z0,jnp.conj(z0),tau0,jnp.conj(tau0),ISDflux_plus,mode="ISD+",output="full",return_integer_flux=True).real
fluxes0,np.sum(np.abs(model.DW(z0,jnp.conj(z0),tau0,jnp.conj(tau0),fluxes0)))

In [ ]:
fluxes0 = sampler.ISD_sampling(z0,jnp.conj(z0),tau0,jnp.conj(tau0),ISDflux_minus,mode="ISD-",output="full",return_integer_flux=True).real
fluxes0,np.sum(np.abs(model.DW(z0,jnp.conj(z0),tau0,jnp.conj(tau0),fluxes0)))

We notice that we do not always obtain the same choice of `fluxes0` for each version.
This is because the different versions ISD sampling find only the **closest integer flux vector** satisfying the ISD condition
for the given input of moduli values.

### Vectorising ISD sampling

We also provide the optional argument to vectorise ISD sampling accross many input points.
To do so, we set `vmap=True` and `in_axes=(0,0,None)` where the latter implies that we are vectorising over moduli and the axio-dilaton, but not over fluxes.
This gives use

In [ ]:
np.random.seed(1)
moduli,tau = sampler.initial_guesses(100,include_fluxes=False)
fluxes0 = sampler.ISD_sampling(moduli,jnp.conj(moduli),tau,jnp.conj(tau),ISDflux_plus,mode="ISD+",output="full",return_integer_flux=True,
                     in_axes=(0,0,None),vmap=True)

fluxes0[:5]

Expected, certain components of the flux vector are always constant because they are specified by `ISDflux_plus=(-2.,  3.,  2.,  0.,  0.,  0.)`.

Alternatively, we can also evaluate the same function accross many different input fluxes as follows:

In [ ]:
np.random.seed(1)
moduli,tau,ISDfluxes = sampler.initial_guesses(100,include_fluxes=True)
fluxes0 = sampler.ISD_sampling(moduli,jnp.conj(moduli),tau,jnp.conj(tau),ISDfluxes[:,:model.n_fluxes],mode="ISD+",output="full",return_integer_flux=True,
                     in_axes=(0,0,0),vmap=True)

fluxes0[:5]

## Summary

This notebook covered the `data_sampler` class and the four ISD sampling modes. Key takeaways:

- `data_sampler` handles moduli, axio-dilaton, and flux sampling with configurable bounds.
- The moduli sampling modes (`"cone"`, `"stretched_cone"`, etc.) control how $\mathrm{Im}(z^i)$ is drawn relative to the mirror Kähler cone.
- ISD sampling replaces half the fluxes by solving the linearised ISD condition, giving starting guesses very close to actual vacua.
- After rounding to integer fluxes, Newton refinement (see notebooks [7](7_ISD_sampling_flux_vacua.ipynb), [8](8_ISD_sampling_wrapper.ipynb)) finds the exact SUSY minimum.

**Next steps:**
- [7_ISD_sampling_flux_vacua](7_ISD_sampling_flux_vacua.ipynb) — single-vacuum ISD workflow with Newton refinement
- [8_ISD_sampling_wrapper](8_ISD_sampling_wrapper.ipynb) — large-scale sampling using `sample_SUSY_flux_vacua`
- [9_sampling_vacua_from_fluxes](9_sampling_vacua_from_fluxes.ipynb) — flux-first approach: find all vacua for given input fluxes
- [10b_stochastic_flux_search](../03_flux_bounding/10b_stochastic_flux_search.ipynb) — stochastic search over bounded flux lattice